## Changing the data from .csv.zst to .delta

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import date_trunc, col

# set schema
schema = StructType([
    StructField("mmsi", LongType(), True),
    StructField("base_date_time", TimestampType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("sog", DoubleType(), True),
    StructField("cog", DoubleType(), True),
    StructField("heading", DoubleType(), True),
    StructField("vessel_name", StringType(), True),
    StructField("imo", StringType(), True),
    StructField("call_sign", StringType(), True),
    StructField("vessel_type", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("length", DoubleType(), True),
    StructField("width", DoubleType(), True),
    StructField("draft", DoubleType(), True),
    StructField("cargo", IntegerType(), True),
    StructField("transceiver", StringType(), True),
])
# read .csv.zst
df = spark.read.option("header", "true")\
    .schema(schema)\
    .csv('/Volumes/workspace/default/raw_data/ais-*.csv.zst')
# create month column for partitions
df = df.withColumn("month", date_trunc("month", "base_date_time").cast("date"))

# write to delta
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("month") \
    .save('/Volumes/workspace/default/raw_data/ais_bronze_delta')

## Comparing read time for .csv.zst to delta

In [0]:
import time

# CSV
start = time.time()
df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "/Volumes/workspace/default/raw_data/ais-*.csv.zst"
)
df_csv.count()
print(f"CSV: {time.time() - start:.1f}s")

# Delta
start = time.time()
df_bronze = spark.read.format("delta").load("/Volumes/workspace/default/raw_data/ais_bronze_delta")
df_bronze.count()
print(f"Delta: {time.time() - start:.1f}s")

CSV: 1703.2s
Delta: 2.1s


## Get Count of Bronze Rows

In [0]:
print(df_delta.count())

3021153531


## Creating Silver Layer

- Thin out un-needed columns
- Do temporal thinning

Not sure show much either of these will matter now that data is in delta format, but I do this when working with python...

In [0]:
from pyspark.sql.functions import date_trunc, row_number, col
from pyspark.sql import Window

# Read bronze
df = spark.read.format("delta").load("/Volumes/workspace/default/raw_data/ais_bronze_delta")

# Column thinning
df = df.drop("imo", "call_sign", "length", "width", "draft")

# Temporal thinning - one point per hour per vessel
window = Window.partitionBy("mmsi", date_trunc("hour", "base_date_time")).orderBy("base_date_time")
df = df.withColumn("rank", row_number().over(window)).filter(col("rank") == 1).drop("rank")

# Write silver
df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("month") \
    .save("/Volumes/workspace/default/raw_data/ais_silver_delta")

# Read Silver and print schema
df_silver = spark.read.format("delta").load('/Volumes/workspace/default/raw_data/ais_silver_delta')
df_silver.printSchema()

root
 |-- mmsi: long (nullable = true)
 |-- base_date_time: timestamp (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- sog: double (nullable = true)
 |-- cog: double (nullable = true)
 |-- heading: double (nullable = true)
 |-- vessel_name: string (nullable = true)
 |-- vessel_type: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- cargo: integer (nullable = true)
 |-- transceiver: string (nullable = true)
 |-- month: date (nullable = true)



## Read Silver and Get Count of Rows

In [0]:
df_silver = spark.read.format("delta").load("/Volumes/workspace/default/raw_data/ais_silver_delta")
print(df_silver.printSchema())
print(df_silver.count())

root
 |-- mmsi: long (nullable = true)
 |-- base_date_time: timestamp (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- sog: double (nullable = true)
 |-- cog: double (nullable = true)
 |-- heading: double (nullable = true)
 |-- vessel_name: string (nullable = true)
 |-- vessel_type: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- cargo: integer (nullable = true)
 |-- transceiver: string (nullable = true)
 |-- month: date (nullable = true)

None
133710106
